In [ ]:
import pandas as pd
import numpy as np

In [ ]:
anime = pd.read_csv("anime-dataset-2023.csv")

In [ ]:
anime.info()

Model - 1 : Content based Recommendation

In [ ]:
required_columns = ["anime_id", "Name", "English name", "Genres", "Synopsis", "Type", "Studios", "Source", "Image URL"]

df = anime[required_columns]
df.info()

In [ ]:
df["Type"].value_counts()

In [ ]:
df = df[df["Type"] != "UNKNOWN"]
df["Type"].unique()

In [ ]:
df['Genres'] = df['Genres'].replace(",", " ")

In [ ]:
df["combined_data"] = df['Genres'] + df["Type"] + df["Studios"]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df["combined_data"])

In [ ]:
tfidf_matrix.shape

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

nn.fit(tfidf_matrix)

In [ ]:
anime_index = 10

distances, indices = nn.kneighbors(
    tfidf_matrix[anime_index],
    n_neighbors=10
)

print(indices)

In [ ]:
for i in indices[0]:
    print(df['English name'].iloc[i])

In [ ]:
title_to_index = pd.Series(df.index, index=df["Name"].str.lower()).to_dict()
title_to_index

In [ ]:
%pip install rapidfuzz

In [ ]:
from rapidfuzz import process, fuzz

def get_best_match(title, title_to_index, score_cutoff=70):
    title = title.lower().strip()

    # Exact match
    if title in title_to_index:
        return title

    # Fuzzy match
    match = process.extractOne(
        title,
        title_to_index.keys(),
        scorer=fuzz.WRatio,
        score_cutoff=score_cutoff
    )

    if match:
        matched_title, score, _ = match
        print(f"Anime  found. Using '{matched_title}' ({score:.1f}% match)")
        return matched_title

    return None

In [ ]:
def recommend(title, top_n=10):
    matched_title = get_best_match(title, title_to_index)

    if matched_title is None:
        print("Anime not found.")
        return []

    idx = title_to_index[matched_title]

    distances, indices = nn.kneighbors(
        tfidf_matrix[idx],
        n_neighbors=top_n + 1
    )

    recommendations = []

    for anime_idx in indices[0][1:]:
        recommendations.append(df.iloc[anime_idx]["Name"])

    return recommendations

   

In [ ]:
print(recommend("narut",20))

Model -2 : Collaberative Recommendation

In [ ]:
ratings = pd.read_csv("users-score-2023.csv")

In [ ]:
ratings.info()

In [ ]:
req_df = ratings[["user_id","anime_id","rating"]]
req_df.shape

In [ ]:
req_df.isnull().sum()

In [ ]:
user_counts = req_df.groupby("user_id").size()
user_counts.describe()

In [ ]:
for t in [20, 30, 50, 75, 90, 100, 122]:
    print(f"{t}: {(user_counts >= t).sum()} users")

In [ ]:
for t in [150, 200, 300, 400, 500, 750, 1000]:
    print(f"{t}: {(user_counts >= t).sum()} users")

In [ ]:
active_users = user_counts[user_counts >= 300].index
filtered_df = req_df[req_df["user_id"].isin(active_users)]

filtered_df.shape

In [ ]:
anime_counts = filtered_df.groupby("anime_id").size()
anime_counts

In [ ]:
anime_counts.describe()

In [ ]:
for t in [10, 25, 50, 100, 250, 500, 1000]:
    print(f"{t}: {(anime_counts >= t).sum()} anime")

In [ ]:
active_anime = anime_counts[anime_counts >= 250].index

filtered_df = filtered_df[filtered_df["anime_id"].isin(active_anime)]

filtered_df.shape

In [ ]:
filtered_df.info()

In [ ]:
from scipy.sparse import csr_matrix

user_codes = filtered_df["user_id"].astype("category").cat.codes
anime_codes = filtered_df["anime_id"].astype("category").cat.codes

sparse_matrix = csr_matrix(
    (
        filtered_df["rating"].astype("float32"),
        (anime_codes, user_codes)
    )
)

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100, random_state=42)

anime_embeddings = svd.fit_transform(sparse_matrix)

In [ ]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(n_neighbors=11, algorithm="brute", metric="cosine")

knn.fit(anime_embeddings)

In [ ]:
# Create mapping between embedding index and anime_id
anime_categories = filtered_df["anime_id"].astype("category").cat.categories

code_to_anime = dict(enumerate(anime_categories))
anime_to_code = {anime_id: code for code, anime_id in code_to_anime.items()}

# Create mapping from anime_id to title
anime_id_to_name = df.set_index("anime_id")["Name"].to_dict()

# Create mapping from title to anime_id
name_to_anime_id = df.assign(Name=df["Name"].str.lower().str.strip()).set_index("Name")["anime_id"].to_dict()

In [ ]:
def recommend_anime(title, n_recommendations=10):
    title = title.lower().strip()
    title = get_best_match(title, title_to_index)

    if title is None:
        print("Anime not found.")
        return []

    anime_id = name_to_anime_id[title]

    if anime_id not in anime_to_code:
        print("Anime not present in the collaborative filtering dataset.")
        return []

    query_index = anime_to_code[anime_id]

    distances, indices = knn.kneighbors(
        anime_embeddings[query_index].reshape(1, -1),
        n_neighbors=n_recommendations + 1
    )

    recommendations = []

    for idx in indices[0][1:]:
        rec_anime_id = code_to_anime[idx]
        recommendations.append(anime_id_to_name.get(rec_anime_id, "Unknown"))

    return recommendations

In [ ]:
print(recommend_anime("narut"))

Hybrid Function (Content - based + Collaberative)

In [ ]:
from collections import defaultdict

def hybrid_recommend(title, top_n=10):

    content_results = recommend(title, top_n=20)
    collab_results = recommend_anime(title, n_recommendations=20)

    scores = defaultdict(float)

    # Content weight = 40%
    for rank, anime in enumerate(content_results):
        scores[anime] += (20 - rank) * 0.4

    # Collaborative weight = 60%
    for rank, anime in enumerate(collab_results):
        scores[anime] += (20 - rank) * 0.6

    final = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    print(f"\nHybrid Recommendations for '{title}':\n")

    for anime, score in final[:top_n]:
        image_url = df.loc[df["Name"] == anime, "Image URL"].iloc[0]
        print(f"{anime} {image_url}")

In [ ]:
hybrid_recommend("Naruto",20)